In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/05_model_input/downloaded_scenarios.csv")
df = df.loc[df["sector"] == "Power",:]

/var/folders/df/zghzv05d7xb8t9xy7y5ld_h40000gn/T/ipykernel_23570/3532604135.py:1: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/05_model_input/downloaded_scenarios.csv")


# Compatible scenarios

In [19]:
# the EBITDA is likely to be negative when the values in fom_usd_per_mw_yr are higher
#  than capacity_factor * hours_per_year * scenario_price
incompatible_fixed_cost = df["om_cost_usd_per_mw_per_yr"] > df["scenario_capacity_factor"] * (24*365) * df["scenario_price"]

# the EBITDA is likely to be negative when fuel_price/efficiency > scenario_price .
incompatible_var_cost = df["fuel_price"] / df["efficiency_decimal"] > df["scenario_price"]

In [20]:
df = df.assign(
    incompatible_var_cost = incompatible_var_cost.astype(bool),
    incompatible_fixed_cost = incompatible_fixed_cost.astype(bool),
)



In [21]:
df.loc[df["scenario_type"] == "baseline", ["scenario_provider", "scenario"]].drop_duplicates()

,scenario_provider,scenario
368114,IMAGE 3.2,SSP1-baseline
399793,IMAGE 3.2,SSP2-baseline
1208657,WITCH 5.0,CO_CurPol
1349111,WITCH 5.0,EN_NoPolicy


In [25]:
pd.options.display.max_rows = 1000

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type", "scenario_geography","sector", "technology", 
    "incompatible_var_cost","incompatible_fixed_cost",
    "likely_incompatible_scenario","incompatible_scenario"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "any"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "any"),
)

incompatibility_flagged = incompatibility_flagged.assign(
    likely_incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] | incompatibility_flagged["incompatible_fixed_cost"]),
    incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] & incompatibility_flagged["incompatible_fixed_cost"]),
)


incompatibility_flagged.query("incompatible_fixed_cost == False")

n_techs  \
scenario_provider     scenario                                                    
AIM/CGE 2.2           EN_NPi2020_1200f                                       14   
                      EN_NPi2020_1400                                        14   
                      EN_NPi2020_1400f                                       14   
                      EN_NPi2020_1600                                        14   
                      EN_NPi2020_1600f                                       14   
                      EN_NPi2020_1800                                        14   
                      EN_NPi2020_1800f                                       14   
AIM/CGE-Korea 2.1     CO_2Deg2020                                            11   
                      CO_2Deg2030                                            11   
                      CO_BAU                                                  8   
                      CO_CurPol                                               8   
                      CO_GPP                                                 11   
                      CO_NDCMCS                                              11   
AIM/Hub-Japan 2.1     EN_NP_2025_-1002050                                    11   
                      EN_NP_2025_-302050                                     10   
                      EN_NP_2025_-402050                                     10   
                      EN_NP_2025_-502050                                     11   
                      EN_NP_2025_-602050                                     11   
                      EN_NP_2025_-702050                                     11   
                      EN_NP_2025_-802050                                     11   
                      EN_NP_2025_-902050                                     11   
                      EN_NP_CurPol                                            8   
                      EN_NP_UNDC                                             10   
AIM/Hub-Korea 2.0     EN_NP_2025_-1002050                                    11   
                      EN_NP_2025_-302050                                      8   
                      EN_NP_2025_-402050                                      9   
                      EN_NP_2025_-502050                                     11   
                      EN_NP_2025_-602050                                     11   
                      EN_NP_2025_-702050                                     10   
                      EN_NP_2025_-802050                                     11   
                      EN_NP_2025_-902050                                     10   
                      EN_NP_CurPol                                            8   
                      EN_NP_UNDC                                             10   
EPPA 6                2CNow_Gradual                                           7   
                      Paris1.5C_OptTax                                        7   
                      Paris2C_Gradual                                         7   
                      Paris2C_OptTax                                          7   
                      ParisFlat                                               7   
                      ParisForever                                            7   
                      Ref                                                     7   
GCAM 5.2              NGFS1_Current policies (Hot house world, Rep)           6   
                      NGFS1_Immediate 2C with CDR (Orderly, Rep)              9   
GCAM 5.3              NGFS2_Below 2°C                                        12   
                      NGFS2_Current Policies                                 12   
                      NGFS2_Divergent Net Zero Policies                      12   
                      NGFS2_Nationally Determined Contributions (NDCs)       12   
                      NGFS2_Net-Zero 2050                                    12   
     

In [27]:
df.loc[df["scenario_provider"] == "WITCH 4.6", "scenario"].unique()

array(['DISCRATE_Ref_dr2p', 'DISCRATE_Ref_dr3p', 'DISCRATE_Ref_dr4p',
       'DISCRATE_Ref_dr5p'], dtype=object)